In [1]:
library(SNFtool)

In [2]:
# MRNA MCI
mrna_emci <- read.csv("extdata/all_emci_mrna.csv", header=TRUE, row.names=1)

# Metab EMCI
metab_emci <- read.csv("extdata/all_emci_metab.csv", header=TRUE, row.names=1)

mrna_emci_all <- read.csv("extdata/mrna20k_emci.csv", header=TRUE)

In [3]:
rownames(mrna_emci_all) = rownames(metab_emci)

mrna_emci_all = mrna_emci_all[,c(-1,-2)]

mrna_emci = mrna_emci[,-1]

metab_emci = metab_emci[,-1]


In [4]:
# only in case of full data
mrna_emci = mrna_emci_all

In [5]:
c(dim(mrna_emci), dim(metab_emci))

[1]   201 20032   201   172

In [6]:
Dist1 = (dist2(as.matrix(mrna_emci),as.matrix(mrna_emci)))^(1/2)
Dist2 = (dist2(as.matrix(metab_emci),as.matrix(metab_emci)))^(1/2)

In [7]:
### parameters

## First, set all the parameters:
K = 10;		# number of neighbors, usually (10~30)
alpha = 0.8;  	# hyperparameter, usually (0.3~0.8)
T =10; 	# Number of Iterations, usually (10~20)

In [8]:
W1 = affinityMatrix(Dist1, K, alpha)
W2 = affinityMatrix(Dist2, K, alpha)

In [9]:
W = SNF(list(W1,W2), K, T)

In [10]:
library("RColorBrewer")
my_palette <- colorRampPalette(c("blue","red"))(n = 500)

In [11]:
#### estimate number of clusters
estimationResult = estimateNumberOfClustersGivenGraph(W, 2:15);
estimationResult

$`Eigen-gap best`
[1] 2

$`Eigen-gap 2nd best`
[1] 8

$`Rotation cost best`
[1] 2

$`Rotation cost 2nd best`
[1] 8

In [12]:
# #### make true labels
truelabel = rep(1,201)


In [13]:
truelabel

[1] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 [38] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 [75] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[112] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[149] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[186] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1

In [14]:
## Get a matrix containing the group information 
## for the samples such as the SpectralClustering result and the True label

C = 2                     # number of clusters
group = spectralClustering(W,C);    # the final subtypes information
group

[1] 2 2 2 2 1 2 2 1 2 1 1 1 1 1 1 1 1 1 1 1 1 2 1 1 2 1 1 1 1 2 2 2 2 2 1 2 1
 [38] 1 1 1 1 2 1 1 2 1 2 1 1 2 2 1 2 1 1 1 1 1 2 2 1 2 2 1 2 1 1 1 1 2 1 2 1 2
 [75] 1 2 1 2 2 2 2 1 2 1 2 1 1 1 1 1 1 1 1 1 1 1 1 2 1 1 1 2 2 1 2 1 1 1 2 1 1
[112] 1 1 2 2 1 1 1 1 2 1 1 1 2 2 1 2 2 1 2 2 1 1 2 1 1 1 1 2 1 1 2 1 1 2 2 1 2
[149] 1 1 2 1 1 1 1 1 2 1 1 2 2 1 2 1 2 2 2 1 1 2 2 2 1 2 2 1 1 2 2 1 1 2 1 1 1
[186] 2 2 1 1 2 1 2 1 1 1 2 1 2 2 1 2

In [15]:
subtype_info<-read.csv("extdata/cn_emci_lmci_ad_clin_pseudotime.csv",header=TRUE)
table(subtype_info$SNF)
name_emci1<-subtype_info$PID[which(subtype_info$SNF=="emcisubtype1")]
name_emci2<-subtype_info$PID[which(subtype_info$SNF=="emcisubtype2")]

snflabel=rep(0,201)

snflabel[which(rownames(W)%in% name_emci1)]=1
snflabel[which(rownames(W)%in% name_emci2)]=2
snflabel


          AD           CN emcisubtype1 emcisubtype2 lmcisubtype1 lmcisubtype2 
         339          534          108           93           85          115 

[1] 2 2 2 2 2 2 1 1 2 1 1 1 1 1 1 2 2 2 1 1 1 2 1 1 2 1 1 1 1 2 1 2 1 2 1 1 1
 [38] 1 1 1 1 2 1 1 2 2 2 1 1 2 2 1 2 1 1 1 1 2 2 2 1 2 2 1 2 1 1 2 1 2 1 2 2 2
 [75] 1 2 1 2 2 2 2 2 2 1 1 1 1 1 1 1 2 1 1 1 1 1 1 2 1 1 1 2 2 1 2 2 1 1 2 1 1
[112] 1 1 2 2 1 1 1 2 2 2 1 1 2 2 1 2 2 1 2 2 2 1 2 2 1 1 1 1 1 1 2 1 1 2 2 1 2
[149] 1 1 2 1 1 2 1 1 2 1 1 2 2 1 2 1 2 2 2 2 1 2 2 2 1 2 1 1 2 2 2 2 1 2 2 1 1
[186] 2 2 1 1 2 1 2 1 1 1 2 1 2 2 1 2

In [16]:
M_label=cbind(snflabel,truelabel)
colnames(M_label)=c("spectralClustering","TrueLabel")

M_label_colors=t(apply(M_label,1,getColorsForGroups))
## or choose you own colors for each label, for example:
M_label_colors=cbind("spectralClustering"=getColorsForGroups(M_label[,"spectralClustering"],colors=c("deepskyblue1","seagreen2","orange1")), ### 
"TrueLabel"=getColorsForGroups(M_label[,"TrueLabel"], colors=c("navy", "red1"))) 
#"TrueLabel"=getColorsForGroups(M_label[,"TrueLabel"], colors=c("khaki1"))) ### in the same order as ()

png(filename="figs/RP_2b.png", width=3.25,  height= 3.25, units= "in", res = 600)
displayClustersWithHeatmap(W, snflabel, M_label_colors)
dev.off()

NULL

png 
  2

In [17]:
paste(group, collapse=",")

[1] "2,2,2,2,1,2,2,1,2,1,1,1,1,1,1,1,1,1,1,1,1,2,1,1,2,1,1,1,1,2,2,2,2,2,1,2,1,1,1,1,1,2,1,1,2,1,2,1,1,2,2,1,2,1,1,1,1,1,2,2,1,2,2,1,2,1,1,1,1,2,1,2,1,2,1,2,1,2,2,2,2,1,2,1,2,1,1,1,1,1,1,1,1,1,1,1,1,2,1,1,1,2,2,1,2,1,1,1,2,1,1,1,1,2,2,1,1,1,1,2,1,1,1,2,2,1,2,2,1,2,2,1,1,2,1,1,1,1,2,1,1,2,1,1,2,2,1,2,1,1,2,1,1,1,1,1,2,1,1,2,2,1,2,1,2,2,2,1,1,2,2,2,1,2,2,1,1,2,2,1,1,2,1,1,1,2,2,1,1,2,1,2,1,1,1,2,1,2,2,1,2"

In [18]:
##### get indexes
subtype1 = c(which(group==1))

subtype2 = c(which(group==2))


In [19]:
paste(subtype1, collapse=",")

[1] "5,8,10,11,12,13,14,15,16,17,18,19,20,21,23,24,26,27,28,29,35,37,38,39,40,41,43,44,46,48,49,52,54,55,56,57,58,61,64,66,67,68,69,71,73,75,77,82,84,86,87,88,89,90,91,92,93,94,95,96,97,99,100,101,104,106,107,108,110,111,112,113,116,117,118,119,121,122,123,126,129,132,133,135,136,137,138,140,141,143,144,147,149,150,152,153,154,155,156,158,159,162,164,168,169,173,176,177,180,181,183,184,185,188,189,191,193,194,195,197,200"

In [20]:
paste(subtype2, collapse=",")

[1] "1,2,3,4,6,7,9,22,25,30,31,32,33,34,36,42,45,47,50,51,53,59,60,62,63,65,70,72,74,76,78,79,80,81,83,85,98,102,103,105,109,114,115,120,124,125,127,128,130,131,134,139,142,145,146,148,151,157,160,161,163,165,166,167,170,171,172,174,175,178,179,182,186,187,190,192,196,198,199,201"

In [21]:
c(length(subtype1),length(subtype2))

[1] 121  80